# DetectionForge — CTI-to-Detection Agent
**Kaggle 5-Day AI Agents Capstone | Track: Agents for Business**

A single Gemini-powered agent that turns threat intel into validated, ATT&CK-mapped Sigma/SPL detection rules — with a built-in evaluation harness.

> This notebook is the submission-friendly view. The full modular repo (recommended for the GitHub link) lives alongside it.

## 1. Setup

In [ ]:
!pip install -q google-genai pydantic pysigma pysigma-backend-splunk pysigma-backend-elasticsearch sentence-transformers beautifulsoup4 pandas matplotlib python-dotenv behave pytest
!git clone -q https://github.com/krovix-1902/detectionforge-.git /kaggle/working/detectionforge 2>/dev/null || echo 'repo already present'

In [ ]:
import os, sys

# Make the repo modules importable whether this notebook runs from
# notebook/ or the repo root (or from /kaggle/working after a git clone).
for p in ("..", ".", "/kaggle/working/detectionforge"):
    if os.path.exists(os.path.join(p, "src")):
        sys.path.insert(0, os.path.abspath(p))
        break

# API key — on Kaggle: Add-ons > Secrets > GEMINI_API_KEY, then:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")
except Exception:
    pass  # locally, a .env file with GEMINI_API_KEY works via python-dotenv

os.environ.setdefault("GEMINI_MODEL", "gemini-2.5-flash")
print("key set:", bool(os.environ.get("GEMINI_API_KEY")))

## 2. The agent
One agent + a 4-tool belt (`fetch_cti`, `map_attack`, `validate_sigma`, `convert_rule`), a self-correction loop, structured Pydantic output, rule memory, and a prompt-injection guard.

In [ ]:
from src.agent import DetectionForgeAgent

agent = DetectionForgeAgent()

## 3. Demo — intel in, validated detection out

In [ ]:
sample = (
    "An adversary sent a spearphishing email with a malicious Word attachment. "
    "A VBA macro launched powershell.exe with a base64-encoded command that "
    "downloaded a payload and created a Registry Run key for persistence."
)
result = agent.run(sample)
print(result.rule.sigma_yaml)
print("\nATT&CK:", [m.technique_id for m in result.rule.attack_mappings])
print("Valid:", result.rule.validation_passed)
print("\nSPL:", result.rule.splunk_spl)
print("\nElastic:", result.rule.elastic_query)
print("\nLog-source assumptions:", result.rule.log_source_assumptions)

## 4. (Optional) Full ATT&CK RAG index
Without this step the agent uses a built-in mini technique map. For the full semantic index over all of MITRE ATT&CK Enterprise, download the STIX bundle and build once — it persists to `data/` and is reused automatically.

In [ ]:
# !wget -q https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/enterprise-attack/enterprise-attack.json
# from src.tools import build_attack_index
# build_attack_index("enterprise-attack.json")

## 5. Evaluation — the proof of quality
Score generated rules on syntax, ATT&CK recall, convertibility, and LLM-judged specificity against a labelled ground-truth set.

In [ ]:
from eval.eval_harness import run_eval

df = run_eval()
print(df.to_string(index=False))
print(f"\nAGGREGATE SCORE: {df['overall'].mean():.3f}")
df

## 6. Architecture & write-up
See README.md for the full architecture, the course capabilities demonstrated, honest limitations, and the ADK porting note. Record the 5-minute video walking through: problem → why agents → architecture → live demo → eval results.